In [1]:
#FORESIGHT - Week 2: EDA & Seasonal-Naive Baseline
#Builds on master_dataset.csv from week 1. Produces seasonality findings, top movers, dead stock, and the seasonal-naive baseline forecast every later model must beat.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

PROCESSED_DIR = Path("../data/processed")
REPORTS_DIR = Path("../reports")

master = pd.read_csv(PROCESSED_DIR / "master_dataset.csv", parse_dates=["date"])
master.head()


,date,sku_id,units_sold,revenue,n_transactions,promo_flag,unit_price_x,year,quarter,month,...,is_weekend,season,is_holiday,promo_event,sku_name,category,subcategory,list_price,unit_price_y,brand
0,2024-01-01,SKU00001,10,475.00,4,True,47.50,2024,1,1,...,0,Winter,False,NaN,NutriPlus Cookware Large,Home & Kitchen,Cookware,813.41,619.77,NutriPlus
1,2024-01-01,SKU00002,10,6309.28,4,True,630.93,2024,1,1,...,0,Winter,False,NaN,CrispKing Bread Family Pack,Dairy & Bakery,Bread,70.38,49.57,CrispKing
2,2024-01-01,SKU00003,5,2225.23,3,True,445.05,2024,1,1,...,0,Winter,False,NaN,SoftTouch Notebooks 2L,Stationery & Office,Notebooks,151.28,83.67,SoftTouch
3,2024-01-01,SKU00004,3,653.70,1,False,217.90,2024,1,1,...,0,Winter,False,NaN,SunriseFoods Eggs Large,Dairy & Bakery,Eggs,233.64,156.32,SunriseFoods
4,2024-01-01,SKU00005,17,4282.04,5,True,251.88,2024,1,1,...,0,Winter,False,NaN,SunriseFoods Pest Control Pack of 6,Home Care,Pest Control,138.50,83.46,SunriseFoods


In [3]:
# Brief asks for a weekly forecast, so collapse daily rows to ISO-week totals.
#  ISO (year, week) avoids merging "week 1 of 2024" with "week 1 of 2025".
iso = master["date"].dt.isocalendar()
master["iso_year"] = iso["year"]
master["iso_week"] = iso["week"]

weekly = (
    master.groupby(["sku_id", "category", "subcategory", "iso_year", "iso_week"], as_index=False)
    .agg(
        units_sold=("units_sold", "sum"),
        revenue=("revenue", "sum"),
        promo_flag=("promo_flag", "max"),

    )
)
weekly["week_start"] = pd.to_datetime(
    weekly["iso_year"].astype(str) + weekly["iso_week"].astype(str) + "1",
    format="%G%V%u",
)
weekly = weekly.sort_values(["sku_id", "week_start"]).reset_index(drop=True)
weekly.head()

,sku_id,category,subcategory,iso_year,iso_week,units_sold,revenue,promo_flag,week_start
0,SKU00001,Home & Kitchen,Cookware,2024,1,83,3902.71,True,2024-01-01
1,SKU00001,Home & Kitchen,Cookware,2024,2,24,1027.18,True,2024-01-08
2,SKU00001,Home & Kitchen,Cookware,2024,3,30,1382.77,True,2024-01-15
3,SKU00001,Home & Kitchen,Cookware,2024,4,58,2758.85,True,2024-01-22
4,SKU00001,Home & Kitchen,Cookware,2024,5,10,481.76,False,2024-01-29


In [4]:
# Index (100 = yearly average) answers "is demand above/below normal?" - 
# a raw monthly total mostly just reflects SKU count, which isn't useful.
monthly = master.groupby(master["date"].dt.month)["units_sold"].sum()
month_seasonality = (monthly / monthly.mean() * 100).round(1)
month_seasonality.index.name = "month"
print(month_seasonality)

month
1      80.1
2      73.9
3      96.0
4      91.6
5      94.6
6      92.1
7      94.2
8      94.6
9      91.6
10     93.1
11    147.8
12    150.4
Name: units_sold, dtype: float64


In [5]:
dow = master.groupby(master["date"].dt.day_name())["units_sold"].sum()
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow = dow.reindex(order)
weekday_seasonality = (dow / dow.mean() * 100).round(1)
print(weekday_seasonality)

date
Monday        89.8
Tuesday       90.0
Wednesday     89.2
Thursday      88.5
Friday        90.1
Saturday     126.2
Sunday       126.2
Name: units_sold, dtype: float64


In [6]:
top_movers = (
    weekly.groupby(["sku_id", "category"], as_index=False)["units_sold"]
    .sum()
    .sort_values("units_sold", ascending=False)
    .head(10)
)
top_movers

,sku_id,category,units_sold
229,SKU00230,Dairy & Bakery,6695
242,SKU00243,Personal Care,6550
225,SKU00226,Beverages,6530
59,SKU00060,Apparel & Footwear,6523
199,SKU00200,Beverages,6504
63,SKU00064,Home Care,6488
204,SKU00205,Stationery & Office,6487
98,SKU00099,Apparel & Footwear,6476
51,SKU00052,Home & Kitchen,6468
48,SKU00049,Personal Care,6459


In [7]:
# Look at the LAST 8 weeks only, not lifetime totals - a SKU that sold well
# in 2024 but has gone cold recently is a markdown candidate NOW.
recent_weeks = 8
threshold_units = 5
cutoff = weekly["week_start"].max() - pd.Timedelta(weeks=recent_weeks)
recent = weekly[weekly["week_start"] > cutoff]

recent_totals = recent.groupby(["sku_id", "category"], as_index=False)["units_sold"].sum()
dead_stock = recent_totals[recent_totals["units_sold"] <= threshold_units].sort_values("units_sold")
dead_stock

,sku_id,category,units_sold


In [8]:
# Not causal (promo days may also be seasonally busier) — flag that caveat
# in your memo rather than overclaiming.
promo_avg = master.loc[master["promo_flag"], "units_sold"].mean()
non_promo_avg = master.loc[~master["promo_flag"], "units_sold"].mean()
lift_pct = (promo_avg / non_promo_avg - 1) * 100

print(f"Avg units/day on promo days: {promo_avg:.2f}")
print(f"Avg units/day on non-promo days: {non_promo_avg:.2f}")
print(f"Apparent lift: {lift_pct:.1f}%")

Avg units/day on promo days: 10.56
Avg units/day on non-promo days: 6.40
Apparent lift: 64.8%


In [9]:
weekly.to_csv(PROCESSED_DIR / "weekly_sales.csv", index=False)

# TODO: write your own 3+ plain-language insights here, e.g.:
# - "December is the strongest month, X% above average — plan reorders in November."
# - "Category X has the most volatile week-to-week demand — prioritize for risk scoring."
# - "N SKUs have gone effectively dead in the last 8 weeks — markdown candidates."
print("Saved weekly_sales.csv")

Saved weekly_sales.csv


In [10]:
## Seasonal-Naive Baseline

WEEKS_PER_SEASON = 52   # one year of lag, since we have 2 years of history
FORECAST_HORIZON = 6    # weeks ahead

# Every SKU needs a row for EVERY week, even weeks with zero sales.
# Skip this and a lag-52 lookup can silently grab the WRONG week for any
# SKU with a gap — an easy, invisible bug.
all_weeks = pd.date_range(weekly["week_start"].min(), weekly["week_start"].max(), freq="W-MON")
all_skus = weekly["sku_id"].unique()

full_index = pd.MultiIndex.from_product([all_skus, all_weeks], names=["sku_id", "week_start"])
panel = (
    weekly.set_index(["sku_id", "week_start"])["units_sold"]
    .reindex(full_index, fill_value=0)
    .reset_index()
    .sort_values(["sku_id", "week_start"])
    .reset_index(drop=True)
)
panel.head()

,sku_id,week_start,units_sold
0,SKU00001,2024-01-01,83
1,SKU00001,2024-01-08,24
2,SKU00001,2024-01-15,30
3,SKU00001,2024-01-22,58
4,SKU00001,2024-01-29,10


In [11]:
# groupby().shift() is the key trick: shift() moves values down N rows
# WITHIN each SKU group, so one SKU's history never leaks into another's forecast.
panel["forecast_naive"] = panel.groupby("sku_id")["units_sold"].shift(WEEKS_PER_SEASON)
panel.dropna(subset=["forecast_naive"]).head()

,sku_id,week_start,units_sold,forecast_naive
52,SKU00001,2024-12-30,76,83.0
53,SKU00001,2025-01-06,43,24.0
54,SKU00001,2025-01-13,53,30.0
55,SKU00001,2025-01-20,70,58.0
56,SKU00001,2025-01-27,51,10.0


In [12]:
def wape(actual, forecast):
    """
    Weighted Absolute Percentage Error = sum(|actual-forecast|) / sum(actual)

    Why not MAPE: MAPE divides by EACH actual value, so a SKU that sold 1 unit
    and was forecast at 3 contributes a 200% error — wildly overweighting
    near-zero weeks. WAPE divides by TOTAL actual demand, so high-volume SKUs
    (which matter more to the business) dominate, and it never blows up on
    small numbers. This is why the brief specifies WAPE as the primary metric.
    """
    actual = np.asarray(actual, dtype=float)
    forecast = np.asarray(forecast, dtype=float)
    return np.abs(actual - forecast).sum() / actual.sum()

In [13]:
def rolling_origin_backtest(panel, n_origins=6):
    """
    Pick several 'origin' weeks near the end of the data. For each, score the
    naive forecast's accuracy over the next FORECAST_HORIZON weeks, using
    only data that would have been available at that origin.

    Why not one random train/test split: a random split lets the model 'see'
    weeks before some of its test weeks, which never happens in real
    forecasting and makes accuracy look better than it is. Rolling-origin
    repeats the real forecasting task at multiple points in time instead.
    """
    max_week = panel["week_start"].max()
    latest_possible_origin = max_week - pd.Timedelta(weeks=FORECAST_HORIZON)
    origin_weeks = pd.date_range(end=latest_possible_origin, periods=n_origins, freq="4W-MON")

    results = []
    for origin in origin_weeks:
        horizon_end = origin + pd.Timedelta(weeks=FORECAST_HORIZON)
        window = panel[(panel["week_start"] > origin) & (panel["week_start"] <= horizon_end)]
        window = window.dropna(subset=["forecast_naive"])
        if window.empty:
            continue
        score = wape(window["units_sold"], window["forecast_naive"])
        results.append({
            "origin_week": origin.date(),
            "horizon_weeks": FORECAST_HORIZON,
            "n_sku_weeks_scored": len(window),
            "wape": round(score, 4),
        })
    return pd.DataFrame(results)

backtest = rolling_origin_backtest(panel)
backtest

,origin_week,horizon_weeks,n_sku_weeks_scored,wape
0,2025-06-30,6,1500,0.3393
1,2025-07-28,6,1500,0.3202
2,2025-08-25,6,1500,0.3213
3,2025-09-22,6,1500,0.3089
4,2025-10-20,6,1500,0.2688
5,2025-11-17,6,1500,0.2943


In [14]:
overall_wape = backtest["wape"].mean()
print(f"Average WAPE across {len(backtest)} origins: {overall_wape:.4f}")
print("This is your bar — any Week 3 model must beat this on the same backtest to ship.")

panel.to_csv(PROCESSED_DIR / "baseline_forecast.csv", index=False)
print("Saved baseline_forecast.csv")

Average WAPE across 6 origins: 0.3088
This is your bar — any Week 3 model must beat this on the same backtest to ship.
Saved baseline_forecast.csv
